In [2]:
import os
import numpy as np
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras import layers, models

In [3]:
train_path = r"E:\study\4th year\8th\thesis\datasets\alzheimers multiclass\train"

class_names = sorted([
    d for d in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, d))
])

print(class_names)

image_paths = []
labels = []

for label, class_name in enumerate(class_names):
    class_dir = os.path.join(train_path, class_name)

    for file in os.listdir(class_dir):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_dir, file))
            labels.append(label)

image_paths = np.array(image_paths)
labels = np.array(labels)

print(len(image_paths))

['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
55443


In [4]:
IMG_SIZE = (224,224)
BATCH_SIZE = 32

def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMG_SIZE)

    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)

    return image, label


def make_dataset(paths, labels, training=False):

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    ds = ds.map(load_image,
                num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(len(paths), seed=42)

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [5]:
def create_model():

    base_model = tf.keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224,224,3)
    )

    base_model.trainable = False

    model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.Dropout(0.5),

        layers.Dense(4)

    ])

    model.compile(

        optimizer=tf.keras.optimizers.Adam(1e-3),

        loss=tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),

        metrics=["accuracy"]

    )

    return model

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

accuracies = []
precisions = []
recalls = []
f1s = []

fold = 1

for train_idx, val_idx in skf.split(image_paths, labels):

    print(f"\n========== Fold {fold} ==========\n")

    X_train = image_paths[train_idx]
    y_train = labels[train_idx]

    X_val = image_paths[val_idx]
    y_val = labels[val_idx]

    train_ds = make_dataset(X_train, y_train, training=True)

    val_ds = make_dataset(X_val, y_val)

    model = create_model()

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=20,
        callbacks=[early_stop],
        verbose=1
    )

    predictions = model.predict(val_ds)

    y_pred = np.argmax(predictions, axis=1)

    acc = accuracy_score(y_val, y_pred)

    pre = precision_score(
        y_val,
        y_pred,
        average="weighted",
        zero_division=0
    )

    rec = recall_score(
        y_val,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_val,
        y_pred,
        average="weighted"
    )

    print("Accuracy :", acc)
    print("Precision:", pre)
    print("Recall   :", rec)
    print("F1 Score :", f1)

    accuracies.append(acc)
    precisions.append(pre)
    recalls.append(rec)
    f1s.append(f1)

    fold += 1


========== Fold 1 ==========

Epoch 1/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 883s 576ms/step - accuracy: 0.6365 - loss: 0.8308 - val_accuracy: 0.7062 - val_loss: 0.6682
Epoch 2/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 878s 547ms/step - accuracy: 0.6724 - loss: 0.7247 - val_accuracy: 0.7092 - val_loss: 0.6411
Epoch 3/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 890s 528ms/step - accuracy: 0.6790 - loss: 0.7122 - val_accuracy: 0.7313 - val_loss: 0.6158
Epoch 4/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 841s 517ms/step - accuracy: 0.6821 - loss: 0.7103 - val_accuracy: 0.7207 - val_loss: 0.6188
Epoch 5/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 838s 518ms/step - accuracy: 0.6833 - loss: 0.7115 - val_accuracy: 0.7371 - val_loss: 0.6049
Epoch 6/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 849s 520ms/step - accuracy: 0.6838 - loss: 0.7073 - val_accuracy: 0.7185 - val_loss: 0.6240
Epoch 7/20
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 855s 528ms/step - accuracy: 0.6840 - loss: 0.7026 - val_accuracy: 0.7284 - val_loss: 0.6098
Epoch 8/20
1387/1387 ━━━━━━━━━━

In [ ]:
print("\n========== Final Result ==========\n")

print(f"Accuracy : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")

print(f"Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")

print(f"Recall   : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")

print(f"F1 Score : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")